<a href="https://colab.research.google.com/github/laragon2-cmd/naive_bayes/blob/main/Naive_Bayes_Case_Study.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np

In [114]:
print(X.shape[0], X.shape[1])


100 4


In [3]:
np.random.seed(3)
# Step  1 Simulate data from known distribution

def building_Y(n_test):

  # Prior probabilities (Y = 1), (Y = 2)
  pi1 = np.random.rand()
  pi2 = 1 - pi1
  pi = np.array([pi1, pi2])

  # Lets create some class labels acording to
  # Priors probabilities, so we'll code class 1 as zero
  # and clas 2 as 1

  Y = np.random.choice([0,1], size = n_test, p = [pi1, pi2])
  return Y, pi

def building_X(n_test, Y):

  X = np.zeros((n_test,4))

  # Lets generate the probability of getting ones for each feature
  p = np.random.rand(X.shape[1])

  for i in range(X.shape[0]):
    if Y[i] == 0:
      X[i] = np.random.binomial(1, p)
    else :
      X[i] = np.random.binomial(1, 1-p)

  return X, p


In [7]:
# Step 2 — Build the Bayes classifier using TRUE parameters:
# Since our futures are binary, each density function is just
# a Bernoully probability.
# Notice we already did calculate/define priors probability "pi1, pi2", now
# We should calculate the posterior probability, which is:
# If Xj = 1   f_kj(1) = P(Xj = 1| Y = k)
# If Xj = 0   f_kj(0) = 1 - P(Xj = 1| Y = k)
# p_hat(Xj = 1 | Y = k) = the proportion of all the X = 1
# in the k clas over the total number of observations in the kth class.


def naive_bayes_classifier(x, X, class_cond_probs, pi):
    K = len(pi)                 # number of classes
    features = X.shape[1]       # number of features

    numerators = np.zeros(K)    # stores pi_k * product of f_kj(x_j)

    for k in range(K):
        density_f_kp = 1

        for j in range(features):
            # estimate P(X_j = 1 | Y = k)
            p_hat = class_cond_probs[k,j]

            # Bernoulli likelihood
            if x[j] == 1:
                density_f_kp *= p_hat
            else:
                density_f_kp *= (1 - p_hat)

        numerators[k] = pi[k] * density_f_kp

    denominator = np.sum(numerators)
    posterior = numerators / denominator
    predicted_class = np.argmax(posterior)

    return posterior, predicted_class


In [8]:
n_test = 100
Y, pi = building_Y(n_test)
X, p = building_X(n_test, Y)

from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42
)



In [9]:

# i. A vector of prior probabilities for each label in
print("Prior probabilities:", pi)
print()


# ii Class conditional probabilities P (xk = 1|Y= j),
# for k = 1, . . . , p and all labels j.
class_cond_probs = np.zeros((len(pi), X_train.shape[1]))

for k in range(len(pi)) :

  for j in range(X_train.shape[1]):
    class_cond_probs[k, j] = np.sum(X_train[Y_train == k, j]) / np.sum(Y_train == k)

print("Class conditional probabilities :")
print('P(X = x | Y = 1): ', class_cond_probs[0])
print('P(X = x | Y = 2): ', class_cond_probs[1])

print()

# iii. a vector of posterior probabilities P (Y= j|X= x) calculated from the algorithm.

posterior, predicted_class = naive_bayes_classifier(X_train[0], X_train, class_cond_probs, pi)
print('Posterior probabilities and predicted class for first observation:')
print(posterior)
print(predicted_class)
print()

# iv. Predicted class for every observation in the training set


predicted_classe_for_each_obs = np.array([naive_bayes_classifier(X_train[i], X_train,
                                        class_cond_probs, pi)[1] for i in
                                        range(X_train.shape[0])]
                                         )

print('Predicted class for every observation')
print(predicted_classe_for_each_obs)
print()

# v. the misclassification rate (between 0 and 1) for the  classifier
# in the training set
nbayes_error_trainig = np.mean(Y_train != predicted_classe_for_each_obs)
print("Naive Bayes Error on training set:", nbayes_error_trainig)
print()


# v.i. The misclassification rate (between 0 and 1) for the  classifier
# now on the testing set

predicted_class_testing = np.array([naive_bayes_classifier(X_test[i], X_train,
                                        class_cond_probs, pi)[1] for i in
                                        range(X_test.shape[0])]
                                         )
print('Predicted class on testing set')
print(predicted_class_testing)
print()
print('The actual classes on testing set')
print(Y_test)
print()

nbayes_error_test = np.mean(Y_test != predicted_class_testing)
print("Naive Bayes Error:", nbayes_error_test)



Prior probabilities: [0.72663087 0.27336913]

Class conditional probabilities :
P(X = x | Y = 1):  [0.86666667 0.76666667 0.38333333 0.98333333]
P(X = x | Y = 2):  [0.2  0.35 0.55 0.15]

Posterior probabilities and predicted class for first observation:
[0.99139996 0.00860004]
0

Predicted class for every observation
[0 1 0 1 1 0 0 0 0 1 0 0 0 0 0 0 0 0 0 1 0 1 0 1 0 0 0 1 1 0 0 1 0 0 0 0 1
 1 0 0 1 1 0 0 0 0 1 0 0 0 0 0 0 0 1 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0
 1 0 1 0 0 0]

Naive Bayes Error on training set: 0.05

Predicted class on testing set
[0 0 0 0 1 0 0 0 0 1 1 1 1 1 0 0 0 0 0 1]

The actual classes on testing set
[0 0 0 0 1 0 0 0 0 1 1 1 0 1 0 0 0 0 0 1]

Naive Bayes Error: 0.05
